# EURUSD H4 - MTO v2 Prototypes (Momentum + Mean Reversion)

Ce notebook implemente 2 modules prototypes:

1. **Momentum court terme** sur \($r_t\): \($r_{t+1} = \alpha r_t + \epsilon_t\)
2. **Mean Reversion (MR) haut timeframe** autour des niveaux WTO/MTO:
   \($\Delta X_t = \kappa_i (TO_i - X_t) + \epsilon_t\)

Puis on construit un **cadre statistique** pour verifier la significativite (in-sample + test OOS).

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MetaTrader5 as mt5
import statsmodels.api as sm
from scipy import stats

from metalib.plot_style import use_metalib_style
from metalib.indicators import (
    get_last_monday_6pm_open_ffill,
    get_second_monday_open_ffill,
)

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:,.6f}'.format
use_metalib_style()


In [ ]:
# -------------------------
# Configuration
# -------------------------
SYMBOL = 'EURUSD'
TIMEFRAME = mt5.TIMEFRAME_H1
NUM_BARS = 12_000
CSV_PATH = Path('metalib/data/eurusd/eurusd_h4_latest.csv')
USE_MT5_PULL = True  # True pour forcer un refresh live

MOM_WINDOW = 250          # ~6 semaines H4
MOM_ALPHA_SIG = 0.10

MR_TIMEFRAME = '1D'       # HTF du module MR
MR_WINDOW = 220           # ~1 an de jours de trading
MR_ALPHA_SIG = 0.10
NUM_TRUE_OPEN_LEVELS = 3  # i=0..2 pour WTO/MTO

TEST_FRAC = 0.30
HAC_LAGS = 5


In [ ]:
def pull_mt5_ohlc(symbol: str, timeframe: int, num_bars: int = 10_000):
    if not mt5.initialize():
        raise RuntimeError(f'MT5 initialization failed: {mt5.last_error()}')

    rates = None
    used_symbol = None
    for suffix in ['', '.a', '.b', '.pro', '_SB']:
        test_symbol = f'{symbol}{suffix}'
        pulled = mt5.copy_rates_from_pos(test_symbol, timeframe, 0, num_bars)
        if pulled is not None and len(pulled) > 0:
            rates = pulled
            used_symbol = test_symbol
            break

    if rates is None:
        err = mt5.last_error()
        mt5.shutdown()
        raise RuntimeError(f'No rates returned for {symbol}. Last MT5 error: {err}')

    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s', utc=True)
    df = df[['time', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume']]
    df.sort_values('time', inplace=True)

    mt5.shutdown()
    return df, used_symbol


if USE_MT5_PULL or (not CSV_PATH.exists()):
    df, used_symbol = pull_mt5_ohlc(SYMBOL, TIMEFRAME, NUM_BARS)
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CSV_PATH, index=False)
    print(f'Pulled {len(df):,} bars from {used_symbol}')
    print(f'Saved: {CSV_PATH.resolve()}')
else:
    df = pd.read_csv(CSV_PATH)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df.sort_values('time', inplace=True)
    print(f'Loaded {len(df):,} rows from {CSV_PATH}')

df.tail()

In [ ]:
# Base series H4
ohlc_h4 = df.set_index('time')[['open', 'high', 'low', 'close']].copy()
close_h4 = ohlc_h4['close'].copy()
ret_h4 = np.log(close_h4).diff()

# Daily OHLC pour MTO
ohlc_daily = ohlc_h4.resample('1D').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
}).dropna()

# True opens WTO/MTO (i=0..N-1)
weekly_levels = {
    f'to_w{i}': get_last_monday_6pm_open_ffill(ohlc_h4, ohlc_h4.index, i=i)
    for i in range(NUM_TRUE_OPEN_LEVELS)
}
monthly_levels = {
    f'to_m{i}': get_second_monday_open_ffill(ohlc_daily, ohlc_h4.index, i=i)
    for i in range(NUM_TRUE_OPEN_LEVELS)
}
levels_h4 = pd.concat([pd.DataFrame(weekly_levels), pd.DataFrame(monthly_levels)], axis=1)

display(levels_h4.tail())
print('H4 range:', ohlc_h4.index.min(), '->', ohlc_h4.index.max())
print('Daily bars:', len(ohlc_daily))

In [ ]:
weekly_levels

In [ ]:
def rolling_ols_no_intercept(y: pd.Series, x: pd.Series, window: int, min_n=None):
    """Rolling OLS sans constante: y_t = beta * x_t + eps_t.
    Retourne beta, t-stat, p-value, R2."""
    idx = y.index
    beta = np.full(len(idx), np.nan)
    tval = np.full(len(idx), np.nan)
    pval = np.full(len(idx), np.nan)
    r2 = np.full(len(idx), np.nan)

    if min_n is None:
        min_n = max(40, window // 2)

    for i in range(len(idx)):
        lo = max(0, i - window + 1)
        y_w = y.iloc[lo:i+1]
        x_w = x.iloc[lo:i+1]

        m = y_w.notna() & x_w.notna()
        yv = y_w[m].to_numpy(dtype=float)
        xv = x_w[m].to_numpy(dtype=float)

        n = len(yv)
        if n < min_n:
            continue

        sxx = float(np.dot(xv, xv))
        if sxx <= 1e-12:
            continue

        b = float(np.dot(xv, yv) / sxx)
        resid = yv - b * xv

        dof = n - 1
        if dof <= 0:
            continue

        sse = float(np.dot(resid, resid))
        mse = sse / dof
        se = float(np.sqrt(max(mse / sxx, 1e-16)))

        t = b / se
        p = 2.0 * (1.0 - stats.t.cdf(abs(t), dof))

        ybar = float(np.mean(yv))
        sst = float(np.dot(yv - ybar, yv - ybar))
        rr = np.nan if sst <= 1e-12 else (1.0 - sse / sst)

        beta[i] = b
        tval[i] = t
        pval[i] = p
        r2[i] = rr

    return pd.DataFrame({
        'beta': beta,
        'tval': tval,
        'pval': pval,
        'r2': r2,
    }, index=idx)


# -------------------------
# Module 1: Momentum quality
# r_t = alpha * r_{t-1} + eps_t
# => equivalent au r_{t+1} = alpha r_t + eps_{t+1}
# -------------------------
mom_est = rolling_ols_no_intercept(
    y=ret_h4,
    x=ret_h4.shift(1),
    window=MOM_WINDOW,
)
mom = mom_est.rename(columns={
    'beta': 'alpha',
    'tval': 'alpha_t',
    'pval': 'alpha_p',
    'r2': 'alpha_r2',
})

# Forecast directionnel du prochain retour
mom['mom_forecast'] = mom['alpha'] * ret_h4
mom['mom_score'] = mom['mom_forecast'] * mom['alpha_t'].abs()
mom['mom_signal'] = np.sign(mom['mom_forecast'])
mom.loc[mom['alpha_p'] >= MOM_ALPHA_SIG, 'mom_signal'] = 0.0

display(mom[['alpha', 'alpha_t', 'alpha_p', 'alpha_r2', 'mom_forecast', 'mom_signal']].tail())
print('Last alpha:', float(mom['alpha'].dropna().iloc[-1]))

In [ ]:
mom.plot(figsize=(20, 30), subplots=True, sharex=True, grid=True, legend=True, xlabel='Time', ylabel='Momentum')

In [ ]:
# -------------------------
# Module 2: MR quality sur HTF
# Delta X_t = kappa_i * (TO_i - X_t) + eps_t
# -------------------------
ohlc_htf = ohlc_h4.resample(MR_TIMEFRAME).agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
}).dropna()
close_htf = ohlc_htf['close']

# Reindex des niveaux H4 vers HTF
levels_htf = levels_h4.reindex(close_htf.index, method='ffill')

mr_cols = []
mr_data = {}

for col in levels_htf.columns:
    # Estimation en mode causal (historiques connus):
    # y_t = X_t - X_{t-1}, x_t = TO_{t-1} - X_{t-1}
    y_hist = close_htf.diff()
    x_hist = levels_htf[col].shift(1) - close_htf.shift(1)

    est = rolling_ols_no_intercept(y=y_hist, x=x_hist, window=MR_WINDOW)

    kappa_col = f'{col}_kappa'
    t_col = f'{col}_kappa_t'
    p_col = f'{col}_kappa_p'
    force_col = f'{col}_force'
    force_sig_col = f'{col}_force_sig'

    mr_data[kappa_col] = est['beta']
    mr_data[t_col] = est['tval']
    mr_data[p_col] = est['pval']

    force_now = est['beta'] * (levels_htf[col] - close_htf)
    mr_data[force_col] = force_now
    mr_data[force_sig_col] = force_now.where(est['pval'] < MR_ALPHA_SIG, 0.0)

    mr_cols.append((col, kappa_col, t_col, p_col, force_col, force_sig_col))

mr_htf = pd.DataFrame(mr_data, index=close_htf.index)
force_sig_cols = [c[-1] for c in mr_cols]
mr_htf['mr_force'] = mr_htf[force_sig_cols].sum(axis=1, min_count=1)
mr_htf['mr_signal'] = np.sign(mr_htf['mr_force'])

# Summary des kappas les plus recents
rows = []
for base, k_col, t_col, p_col, _, _ in mr_cols:
    last_idx = mr_htf[k_col].dropna().index.max()
    if pd.isna(last_idx):
        continue
    k = float(mr_htf.loc[last_idx, k_col])
    t = float(mr_htf.loc[last_idx, t_col])
    p = float(mr_htf.loc[last_idx, p_col])
    relation = 'attractor' if k > 0 else 'repulsor'
    rows.append({
        'level': base,
        'kappa': k,
        't_stat': t,
        'p_value': p,
        'relation': relation,
        'signif_10pct': p < MR_ALPHA_SIG,
    })

kappa_summary = pd.DataFrame(rows).sort_values('level')
display(kappa_summary)

# Forward-fill vers H4 pour fusion avec module momentum
mr_h4 = mr_htf[['mr_force', 'mr_signal']].reindex(ohlc_h4.index, method='ffill')
mr_h4.tail()

In [ ]:
close_htf

In [ ]:
def _binom_pvalue_greater(k: int, n: int, p0: float = 0.5):
    if n <= 0:
        return np.nan
    try:
        return stats.binomtest(k, n, p=p0, alternative='greater').pvalue
    except AttributeError:
        return stats.binom_test(k, n, p=p0, alternative='greater')


def eval_signal(signal: pd.Series, next_ret: pd.Series, name: str, hac_lags: int = 5):
    z = pd.concat([signal.rename('signal'), next_ret.rename('next_ret')], axis=1).dropna()
    z = z[z['signal'] != 0]

    if len(z) < 50:
        return {
            'name': name,
            'n_obs': len(z),
            'hit_rate': np.nan,
            'binom_p': np.nan,
            'mean_signed_ret': np.nan,
            't_stat_signed': np.nan,
            't_p_signed': np.nan,
            'spearman_ic': np.nan,
            'hac_beta': np.nan,
            'hac_t': np.nan,
            'hac_p': np.nan,
        }

    signed_ret = np.sign(z['signal']) * z['next_ret']
    hits = (signed_ret > 0).astype(int)

    hit_rate = float(hits.mean())
    k = int(hits.sum())
    n = int(len(hits))
    binom_p = float(_binom_pvalue_greater(k, n, p0=0.5))

    t_stat, t_p = stats.ttest_1samp(signed_ret.dropna(), popmean=0.0)
    ic = float(z['signal'].corr(z['next_ret'], method='spearman'))

    try:
        X = sm.add_constant(z['signal'])
        ols = sm.OLS(z['next_ret'], X).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags})
        beta = float(ols.params['signal'])
        beta_t = float(ols.tvalues['signal'])
        beta_p = float(ols.pvalues['signal'])
    except Exception:
        beta, beta_t, beta_p = np.nan, np.nan, np.nan

    return {
        'name': name,
        'n_obs': n,
        'hit_rate': hit_rate,
        'binom_p': binom_p,
        'mean_signed_ret': float(signed_ret.mean()),
        't_stat_signed': float(t_stat),
        't_p_signed': float(t_p),
        'spearman_ic': ic,
        'hac_beta': beta,
        'hac_t': beta_t,
        'hac_p': beta_p,
    }


# -------------------------
# Fusion des modules + evaluation statistique
# -------------------------
eval_df = pd.DataFrame(index=ohlc_h4.index)
eval_df['next_ret'] = ret_h4.shift(-1)

eval_df['mom_signal'] = mom['mom_signal']
eval_df['mom_score'] = mom['mom_score']

eval_df['mr_signal'] = mr_h4['mr_signal']
eval_df['mr_score'] = mr_h4['mr_force']

# Signal combine: uniquement quand momentum et MR sont alignes
same_side = (np.sign(eval_df['mom_signal']) == np.sign(eval_df['mr_signal'])) & (eval_df['mom_signal'] != 0)
eval_df['combo_signal'] = np.where(same_side, np.sign(eval_df['mom_signal']), 0.0)
eval_df['combo_score'] = eval_df['mom_score'].fillna(0.0) + eval_df['mr_score'].fillna(0.0)

# Split IS / OOS
split_idx = int(len(eval_df) * (1.0 - TEST_FRAC))
is_df = eval_df.iloc[:split_idx].copy()
oos_df = eval_df.iloc[split_idx:].copy()

signal_map = {
    'mom_signal': 'Momentum sign',
    'mr_signal': 'MR force sign',
    'combo_signal': 'Aligned momentum+MR sign',
    'mom_score': 'Momentum continuous score',
    'mr_score': 'MR continuous score',
    'combo_score': 'Combined continuous score',
}

res_is = pd.DataFrame([
    eval_signal(is_df[col], is_df['next_ret'], f'IS | {label}', hac_lags=HAC_LAGS)
    for col, label in signal_map.items()
])

res_oos = pd.DataFrame([
    eval_signal(oos_df[col], oos_df['next_ret'], f'OOS | {label}', hac_lags=HAC_LAGS)
    for col, label in signal_map.items()
])

display(res_is.sort_values('hac_p'))
display(res_oos.sort_values('hac_p'))

In [ ]:
# -------------------------
# Visual diagnostics
# -------------------------
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

axes[0].plot(mom.index, mom['alpha'], label='alpha (momentum)', color='tab:blue')
axes[0].axhline(0, color='black', lw=1, alpha=0.6)
axes[0].set_title('Momentum AR(1) coefficient alpha')
axes[0].legend(loc='upper left')

axes[1].plot(mr_htf.index, mr_htf['mr_force'], label='MR aggregate force (HTF)', color='tab:orange')
axes[1].axhline(0, color='black', lw=1, alpha=0.6)
axes[1].set_title('Aggregate MR force from WTO/MTO levels')
axes[1].legend(loc='upper left')

test_slice = oos_df.copy()
for col, lab in [('mom_signal', 'Momentum'), ('mr_signal', 'MR'), ('combo_signal', 'Aligned combo')]:
    strat_ret = np.sign(test_slice[col]) * test_slice['next_ret']
    axes[2].plot(strat_ret.fillna(0.0).cumsum(), label=lab)
axes[2].set_title('OOS cumulative signed returns (prototype, no costs)')
axes[2].legend(loc='upper left')

plt.tight_layout()
plt.show()

## Lecture rapide des resultats

- **Momentum**: regarder `alpha`, `alpha_t`, `alpha_p` et la stabilite temporelle.
- **MR**: pour chaque niveau `to_wi/to_mi`, le signe de `kappa` indique attracteur (`kappa>0`) ou repulseur (`kappa<0`).
- **Significativite**:
  - `binom_p` sur hit-rate directionnel (>50%)
  - `t_p_signed` sur la moyenne des signed returns
  - `hac_p` sur regression HAC de `next_ret` sur le signal
- **Validation pragmatique**: privilegier OOS (`OOS | ...`) et robustesse sur plusieurs periodes/market regimes avant integration live.